## PARTE 2-PRUEBA SIATA-CIENTÍFICO DE DATOS:

## Tarea A: Análisis exploratorio de datos (EDA)

In [1]:
# ============ TAREA A: EDA por conjunto y comparativo ============
import pandas as pd, numpy as np
from io import StringIO

FILES = {
 "met_367":  ("Estacion_meteorologica_367_2025-12-01_2026-04-30.csv",
              ["h","t","pr","vv","vv_max","dv","dv_max","p"]),
 "niv_803":  ("Estacion_nivel_803_2025-12-01_2026-04-30.csv", ["nivel"]),
 "pir_6004": ("Estacion_piranometro_6004_2025-12-01_2026-04-30.csv", ["radiacion"]),
 "plu_35":   ("Estacion_pluviometrica_35_2025-12-01_2026-04-30.csv", ["p1","p2"]),
}
NOMINAL_MIN = 151*1440  # 1-dic-2025 a 30-abr-2026

def load(key):
    path, vals = FILES[key]
    raw = [l for l in open(path, encoding="utf-8", errors="replace").read().splitlines() if l.strip()]
    ncol = 2 + len(vals) + 1
    good, bad = ([l for l in raw if len(l.split(","))==ncol],
                 [l for l in raw if len(l.split(","))!=ncol])
    df = pd.read_csv(StringIO("\n".join(good)), header=None,
                     names=["codigo","fecha_hora"]+vals+["calidad"])
    df["fecha_hora"] = pd.to_datetime(df["fecha_hora"], errors="coerce")
    for c in vals+["calidad"]: df[c] = pd.to_numeric(df[c], errors="coerce")
    return df.sort_values("fecha_hora").reset_index(drop=True), bad, vals

DATA = {}
for k in FILES: DATA[k] = load(k)

# ---- A1 Perfilamiento ----
print("="*70); print("A1. PERFILAMIENTO")
for k,(df,bad,vals) in DATA.items():
    d = df.fecha_hora.diff().dropna()
    print(f"\n[{k}] filas={len(df)} líneas_malas={len(bad)} esperadas={NOMINAL_MIN}")
    print(f"  rango={df.fecha_hora.min()} → {df.fecha_hora.max()}")
    print(f"  nulos_por_col={df[vals].isna().sum().to_dict()}")
    print(f"  ts_duplicados={df.fecha_hora.duplicated().sum()} filas_dup={df.duplicated().sum()}")
    print(f"  ordenado={df.fecha_hora.is_monotonic_increasing} paso_mediana={d.median()} pasos!=1min={(d!=pd.Timedelta('1min')).sum()}")
    print(f"  calidad_codes={df.calidad.value_counts().head(6).to_dict()}")
    for c in vals:
        vc = df[c].round(2).value_counts().head(3)
        print(f"  {c}: min={df[c].min():.3f} max={df[c].max():.3f} top_repetidos={vc.to_dict()}")

# ---- A2 Completitud temporal ----
print("="*70); print("A2. HUECOS")
for k,(df,bad,vals) in DATA.items():
    idx = pd.date_range(df.fecha_hora.min(), df.fecha_hora.max(), freq="1min")
    miss = idx.difference(df.fecha_hora)
    if len(miss)==0: print(f"[{k}] sin huecos"); continue
    s = pd.Series(miss); brk = (s.diff() > pd.Timedelta("1min")).cumsum()
    runs = s.groupby(brk).agg(ini="min", fin="max", n="size")
    print(f"[{k}] huecos_min={len(miss)} ({100*len(miss)/len(idx):.2f}%) n_eventos={len(runs)} "
          f"dur_max={runs.n.max()} dur_med={runs.n.mean():.1f}")
    print(f"   por_hora_dia(top3)={s.dt.hour.value_counts().head(3).to_dict()}")
    print(f"   por_mes={s.dt.month.value_counts().sort_index().to_dict()}")

# ---- A3 calidad vs valores ----
print("="*70); print("A3. CALIDAD vs VALORES")
for k,(df,bad,vals) in DATA.items():
    g = df.groupby("calidad")[vals].agg(["mean","max","count"])
    print(f"\n[{k}]"); print(g.round(3).to_string())

# ---- A4 ciclos ----
print("="*70); print("A4. CICLOS (resumen horario)")
for k,(df,bad,vals) in DATA.items():
    h = df.set_index("fecha_hora")[vals[0]]
    hr = h.resample("1h").mean()
    print(f"[{k}] diurno(max-min por hora)={hr.groupby(hr.index.hour).mean().pipe(lambda x: x.max()-x.min()):.2f} "
          f"| semanal=cv_por_dow={hr.groupby(hr.index.dayofweek).mean().std():.3f} "
          f"| mensual={hr.groupby(hr.index.month).mean().round(2).to_dict()}")

# ---- A5 consistencia física ----
print("="*70); print("A5. CONSISTENCIA FÍSICA")
df,_ ,_ = DATA["met_367"]
print("[367] h_fuera[0,100]=", int(((df.h<0)|(df.h>100)).sum()),
      "| t_fuera[-5,40]=", int(((df.t<-5)|(df.t>40)).sum()),
      "| pr_fuera[700,900]=", int(((df.pr<700)|(df.pr>900)).sum()),
      "| vv_max<vv=", int((df.vv_max<df.vv).sum()),
      "| dv_fuera[0,360]=", int(((df.dv<0)|(df.dv>360)).sum()),
      "| p<0=", int((df.p<0).sum()))
df,_,_ = DATA["niv_803"]
dj = df.nivel.diff().abs()
print("[803] salto>0.5m/min=", int((dj>0.5).sum()),
      "| mesetas>=10min=", int((df.nivel.diff().eq(0).rolling(10).sum()>=10).sum()),
      "| nivel<0=", int((df.nivel<0).sum()))
df,_,_ = DATA["pir_6004"]
noche = df.fecha_hora.dt.hour.isin(list(range(19,24))+list(range(0,6)))
print("[6004] rad<0=", int((df.radiacion<0).sum()),
      "| rad>1300=", int((df.radiacion>1300).sum()),
      "| rad>50_de_noche=", int(((df.radiacion>50)&noche).sum()))
df,_,_ = DATA["plu_35"]
print("[35] p<0=", int(((df.p1<0)|(df.p2<0)).sum()),
      "| no_multiplo_0.254=", int(((df.p1%0.254>1e-6)&(df.p1>0)).sum()),
      "| |p1-p2|>0.254=", int((abs(df.p1-df.p2)>0.254).sum()))

C:\Users\SEBAS GJ\AppData\Local\Temp\ipykernel_584\1810669514.py:20: DtypeWarning: Columns (0,2,3,4,5,6,7,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(StringIO("\n".join(good)), header=None,
C:\Users\SEBAS GJ\AppData\Local\Temp\ipykernel_584\1810669514.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["fecha_hora"] = pd.to_datetime(df["fecha_hora"], errors="coerce")
C:\Users\SEBAS GJ\AppData\Local\Temp\ipykernel_584\1810669514.py:20: DtypeWarning: Columns (0,2,3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(StringIO("\n".join(good)), header=None,
C:\Users\SEBAS GJ\AppData\Local\Temp\ipykernel_584\1810669514.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing 

A1. PERFILAMIENTO

[met_367] filas=217398 líneas_malas=0 esperadas=217440
  rango=2025-12-01 00:00:00 → 2026-04-30 23:59:00
  nulos_por_col={'h': 1, 't': 1, 'pr': 1, 'vv': 1, 'vv_max': 1, 'dv': 1, 'dv_max': 1, 'p': 1}
  ts_duplicados=0 filas_dup=0
  ordenado=False paso_mediana=0 days 00:01:00 pasos!=1min=22
  calidad_codes={1.0: 216712, 1511.0: 508, 154.0: 105, 1567.0: 72}
  h: min=35.940 max=100.000 top_repetidos={94.0: 8327, 92.0: 7973, 93.0: 7710}
  t: min=13.450 max=28.600 top_repetidos={16.9: 3487, 17.2: 3466, 16.7: 3272}
  pr: min=809.510 max=821.800 top_repetidos={816.6: 3191, 816.2: 3139, 816.0: 3105}
  vv: min=0.000 max=13.470 top_repetidos={0.38: 2458, 0.41: 2437, 0.44: 2425}
  vv_max: min=0.000 max=14.300 top_repetidos={0.7: 18890, 0.6: 17995, 0.8: 17386}
  dv: min=0.000 max=359.000 top_repetidos={283.0: 1482, 282.0: 1477, 285.0: 1429}
  dv_max: min=0.000 max=360.000 top_repetidos={277.0: 3266, 280.0: 2535, 274.0: 2475}
  p: min=0.000 max=0.520 top_repetidos={0.0: 207849, 0.

## A6 — Tabla de hallazgos (pega y ajusta números con los prints):

| # | Problema | Evidencia (reproducible) | Impacto | Recomendación |
|---|---|---|---|---|
| 1 | Completitud desigual entre sensores | 6004 pierde ~2.616 min (98,8%); 803 ~122; 367 ~43; 35 completo | Sesgo en agregados y alertas si se asume serie completa | Publicar completitud por estación (Indicador 1, P1.3) y no imputar en silencio |
| 2 | Códigos de calidad no documentados | `calidad` toma valores 1 y códigos de 4 dígitos (1511, 1512, 1583) sin diccionario | Imposible auditar sin reconstruir significado | Publicar diccionario de banderas versionado (ISO 19115 / OMM 1131) |
| 3 | Mesetas (sensor congelado) en nivel 803 | Repetición de 18,10 m por ≥10 min consecutivos | Subestima variabilidad y dispara falsos "estables" | Regla de persistencia con bandera 2 (P1.6 etapa 3) |
| 4 | Radiación nocturna no nula en 6004 | radiacion > 50 W/m² entre 19:00 y 06:00 | Contamina balances de energía y STL | Regla de rango dependiente de elevación solar (bandera 3) |
| 5 | Cuantización float32 en nivel 803 | Valores tipo 6,0999755859375 (paso ~0,05 m) | Límite de exactitud en umbrales de creciente | Declarar resolución en metadatos; redondear a 0,01 m al publicar |
| 6 | Doble canal p1/p2 con discrepancias en 35 | |p1−p2| = 0,254 en minutos secos | Dos versiones del mismo fenómeno | Regla de coherencia interna; usar promedio o canal maestro documentado |
| 7 | vv_max < vv en meteorológica 367 | Conteos > 0 en A5 | Inconsistencia física de ráfaga vs media | Regla de coherencia vv_max ≥ vv (bandera 2) |

## Tarea B: Modelado de series de tiempo

In [5]:
# ================= TAREA B — CONSOLIDADO DEPURADO (B1–B6) =================
import os, warnings
import numpy as np, pandas as pd
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, acf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error as MAE, mean_squared_error as MSE
warnings.filterwarnings("ignore")
os.makedirs("outputs", exist_ok=True)

H, N_ORIG, TRAIN_DAYS = 24, 3, 30      # horizonte (h), orígenes móviles, días de train
                                        # (si el reloj aprieta: N_ORIG=2, TRAIN_DAYS=21)

# ---- B1: una variable por archivo, con justificación ----
SERIES = {
 "t_367":     ("met_367",  "t",         "mean", "Temperatura: continua, ciclo diurno fuerte, baja intermitencia"),
 "nivel_803": ("niv_803",  "nivel",     "mean", "Nivel: continua con respuesta a lluvia y baseflow suave"),
 "rad_6004":  ("pir_6004", "radiacion", "mean", "Radiación: ciclo diurno determinista con cero nocturno"),
 "lluvia_35": ("plu_35",   "p1",        "sum",  "Lluvia: intermitente y cero-inflada (caso especial B3)"),
}

def prep(key, col, agg):
    """B1: máscara por calidad, regularización horaria y tratamiento de huecos."""
    df, _, _ = DATA[key]
    d  = df.set_index("fecha_hora")
    ok = d["calidad"].eq(1)                 # calidad != 1 → valor sospechoso → faltante
    s  = d[col].where(ok)
    n_flag = int((~ok).sum())
    s  = s.resample("1h").agg(agg).asfreq("1h")
    n_gap = int(s.isna().sum())
    if agg == "sum":
        s = s.fillna(0)                     # lluvia: hueco de telemetría = 0 mm (supuesto documentado)
    else:
        s = s.interpolate(limit=6).ffill().bfill()   # continuas: interpolación acotada ≤6 h
    return s, n_flag, n_gap

def feats(s, H=24):
    """B3: features SIN fuga: solo lags >= H (conocidos en el origen) + calendario."""
    X = pd.DataFrame(index=s.index)
    for L in [24, 25, 48, 72, 168]:
        X[f"lag{L}"] = s.shift(L)
    X["hour"], X["dow"], X["month"] = s.index.hour, s.index.dayofweek, s.index.month
    return X

def mase_scale(v, m=24):
    """Escala MASE correcta: media de |y_t - y_{t-m}| sobre el entrenamiento."""
    d = np.abs(v[m:] - v[:-m])
    return float(d.mean()) if d.mean() > 1e-9 else 1e-9

def eval_rolling(s, name):
    """B4: rolling-origin sin fuga; naive estacional, SARIMA y GBM con rezagos."""
    train_n = TRAIN_DAYS * 24
    start   = len(s) - train_n - N_ORIG * H
    rows = []
    for o in range(N_ORIG):
        tr_end   = start + train_n + o * H
        ytr, yte = s.iloc[:tr_end], s.iloc[tr_end:tr_end + H]
        sc       = mase_scale(ytr.values, H)
        p_naive  = s.shift(H).iloc[tr_end:tr_end + H].values      # naive estacional (referencia)
        try:
            m = SARIMAX(ytr, order=(1,1,1), seasonal_order=(1,0,1,H),
                        enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
            p_sar = np.asarray(m.forecast(H))
        except Exception:
            p_sar = p_naive.copy()                                # lluvia puede no converger
        X   = feats(s, H)
        Xtr = X.iloc[:tr_end].dropna()
        g   = HistGradientBoostingRegressor(max_iter=150, random_state=0)
        g.fit(Xtr, ytr.loc[Xtr.index])
        p_gbm = g.predict(X.iloc[tr_end:tr_end + H])
        for mn, p in [("naive_est", p_naive), ("SARIMA", p_sar), ("GBM_lags", p_gbm)]:
            e = yte.values - p
            rows.append([name, o, mn, MAE(yte, p), np.sqrt(MSE(yte, p)), MAE(yte, p) / sc])
    return pd.DataFrame(rows, columns=["serie","origen","modelo","MAE","RMSE","MASE"])

# ---- B1 + B2: preproceso, descomposición STL, estacionariedad y autocorrelación ----
S = {}
for nm, (k, c, a, just) in SERIES.items():
    s, n_flag, n_gap = prep(k, c, a); S[nm] = s
    stl = STL(s, period=24, robust=True).fit()
    adf = adfuller(s.diff(24).dropna())[1]
    ac  = acf(s.diff(24).dropna(), nlags=48)
    print(f"[{nm}] {just}")
    print(f"  B1: minutos_flag_calidad={n_flag} | huecos_horarios={n_gap} | len_serie={len(s)}")
    print(f"  B2: var={s.var():.2f} tend_amp={stl.trend.max()-stl.trend.min():.2f} "
          f"est_amp={stl.seasonal.max()-stl.seasonal.min():.2f} "
          f"ADF(diff24)_p={adf:.2g} ACF_lag24={ac[24]:.2f}")

# ---- B3 + B4: comparación de enfoques con validación de origen móvil ----
R = pd.concat([eval_rolling(s, nm) for nm, s in S.items()])
R.to_csv("outputs/tabla_B4_rolling_origin.csv", index=False)
G = R.groupby(["serie","modelo"])[["MAE","RMSE","MASE"]].mean().reset_index()
print("\n==== B4: rolling-origin, promedio por serie y modelo ====")
print(G.round(3).to_string(index=False))

# ---- B5: incertidumbre con cuantiles empíricos de residuos (GBM, t_367) ----
s = S["t_367"]; X = feats(s, H); tr = len(s) - 240
Xtr = X.iloc[:tr].dropna()
g = HistGradientBoostingRegressor(max_iter=150, random_state=0).fit(Xtr, s.iloc[:tr].loc[Xtr.index])
pred  = g.predict(X.iloc[tr:])
resid = s.iloc[tr:].values - pred
q10, q90 = np.quantile(resid, [0.1, 0.9])
cov = float(np.mean((s.iloc[tr:].values >= pred + q10) & (s.iloc[tr:].values <= pred + q90)))
print(f"\nB5 t_367: intervalo empírico 80% = [{q10:.2f}, {q90:.2f}] | cobertura observada = {cov:.2f} (objetivo 0.80)")

# ---- B6: mejor modelo por serie (insumo de la conclusión operativa) ----
best = G.loc[G.groupby("serie")["MAE"].idxmin()]
print("\n==== B6: mejor modelo por serie (menor MAE) ====")
print(best.round(3).to_string(index=False))

[t_367] Temperatura: continua, ciclo diurno fuerte, baja intermitencia
  B1: minutos_flag_calidad=686 | huecos_horarios=0 | len_serie=3624
  B2: var=9.05 tend_amp=5.23 est_amp=12.36 ADF(diff24)_p=4.2e-22 ACF_lag24=-0.35
[nivel_803] Nivel: continua con respuesta a lluvia y baseflow suave
  B1: minutos_flag_calidad=26277 | huecos_horarios=116 | len_serie=3624
  B2: var=3.80 tend_amp=7.80 est_amp=12.92 ADF(diff24)_p=3.2e-22 ACF_lag24=-0.45
[rad_6004] Radiación: ciclo diurno determinista con cero nocturno
  B1: minutos_flag_calidad=1732 | huecos_horarios=31 | len_serie=3624
  B2: var=97861.45 tend_amp=305.41 est_amp=1343.22 ADF(diff24)_p=7.9e-26 ACF_lag24=-0.45
[lluvia_35] Lluvia: intermitente y cero-inflada (caso especial B3)
  B1: minutos_flag_calidad=3053 | huecos_horarios=0 | len_serie=3624
  B2: var=2.62 tend_amp=3.33 est_amp=43.46 ADF(diff24)_p=2.1e-30 ACF_lag24=-0.49

==== B4: rolling-origin, promedio por serie y modelo ====
    serie    modelo    MAE    RMSE  MASE
lluvia_35  GBM_la

### Conclusiones y hallazgos:

- **Recomendación operativa:** GBM con rezagos ≥24 h y calendario para temperatura,
nivel y radiación (menor MAE y MASE < 1 vs naive estacional); para lluvia, el GBM
domina pero con MASE inestable por cero-inflación → se recomienda modelo hurdle
(clasificador seco/húmedo + regresor en húmedos) o Croston en producción.

- **Incertidumbre:** intervalos empíricos de residuos dan cobertura ≈80% en temperatura;
en lluvia la cobertura se degrada → usar cuantiles condicionales por régimen.

- **Supuestos bajo los cuales fallaría:** cambio de régimen (mantenimiento o
recalibración del sensor), huecos largos de telemetría (>6 h), eventos extremos
fuera del soporte histórico, y deriva de calibración no banderada en `calidad`.

###  Parte C — Detección de anomalías:

In [6]:
# ============ TAREA C — CONSOLIDADO DEPURADO (C1–C5) ============
import pandas as pd, numpy as np
from statsmodels.tsa.seasonal import STL
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score

# ---- C1: reglas deterministas por tipo de sensor ----
def reglas(df, tipo):
    f = pd.DataFrame(index=df.index)
    if tipo == "meteo":
        f["R_rango_h"]  = (df.h < 0) | (df.h > 100)
        f["R_rango_t"]  = (df.t < -5) | (df.t > 40)
        f["R_rango_pr"] = (df.pr < 700) | (df.pr > 900)
        f["R_vvmax"]    = df.vv_max < df.vv
        f["R_rango_dv"] = (df.dv < 0) | (df.dv > 360) | (df.dv_max < 0) | (df.dv_max > 360)
        f["R_pneg"]     = df.p < 0
    elif tipo == "nivel":
        f["R_salto"]    = df.nivel.diff().abs() > 0.5
        f["R_meseta"]   = (df.nivel.diff().eq(0).rolling(10).sum() >= 10)
        f["R_neg"]      = df.nivel < 0
        f["R_rango"]    = (df.nivel < -2) | (df.nivel > 30)
    elif tipo == "piran":
        noche = df.fecha_hora.dt.hour.isin(list(range(19, 24)) + list(range(0, 6)))
        f["R_noche"]    = (df.radiacion > 50) & noche
        f["R_neg"]      = df.radiacion < 0
        f["R_alta"]     = df.radiacion > 1300
    elif tipo == "pluvio":
        f["R_neg"]      = (df.p1 < 0) | (df.p2 < 0)
        f["R_disc"]     = (df.p1 - df.p2).abs() > 0.254
    out = f.fillna(False)
    out["reglas"] = out.any(axis=1).astype(int)
    return out

# ---- C2a: residuos STL + MAD (horario → minutal alineado POR POSICIÓN) ----
def stl_mad_flags(df, col):
    s   = df.set_index("fecha_hora")[col].resample("1h").mean()
    s   = s.interpolate(limit=6).ffill().bfill()
    stl = STL(s, period=24, robust=True).fit()
    res = stl.resid
    mad = 1.4826 * np.nanmedian(np.abs(res - np.nanmedian(res)))
    flag_h   = (res.abs() > 3 * mad).astype(int)
    flag_min = (flag_h.reindex(df.set_index("fecha_hora").index, method="ffill")
                      .fillna(0).astype(int).to_numpy())   # FIX: posición, no índice
    return flag_min, stl, mad

# ---- C2b: Isolation Forest (submuestreo para velocidad en ventana de examen) ----
def iso_flags(df, cols):
    X = df[cols].fillna(0).to_numpy()
    idx = np.random.default_rng(0).choice(len(X), size=min(60000, len(X)), replace=False)
    iso = IsolationForest(n_estimators=100, contamination=0.02, random_state=0).fit(X[idx])
    return (iso.predict(X) == -1).astype(int)

TIPO = {"met_367":  ("meteo",  "t",         ["h","t","pr","vv","vv_max","dv"]),
        "niv_803":  ("nivel",  "nivel",     ["nivel"]),
        "pir_6004": ("piran",  "radiacion", ["radiacion"]),
        "plu_35":   ("pluvio", "p1",        ["p1","p2"])}

FLAGS = {}
for k, (tipo, col, cols) in TIPO.items():
    df, _, _ = DATA[k]
    R = reglas(df, tipo)
    S, stl, mad = stl_mad_flags(df, col)
    M = iso_flags(df, cols)
    FLAGS[k] = pd.DataFrame({                      # FIX: todo por posición (.to_numpy())
        "fecha_hora": df.fecha_hora.to_numpy(),
        "reglas":     R["reglas"].to_numpy(),
        "stl_mad":    S,
        "iso":        M,
        "calidad":    df.calidad.to_numpy(),
    })
    print(f"[{k}] reglas={int(R['reglas'].sum())} stl_mad={int(S.sum())} "
          f"iso={int(M.sum())} calidad!=1={int((df.calidad != 1).sum())}")

# ---- C3: reglas vs estadístico vs ML, contra calidad como ETIQUETA DÉBIL ----
print("\n==== C3: precisión / exhaustividad / F1 vs calidad (débil) ====")
rows = []
for k, F in FLAGS.items():
    y = (F["calidad"].values != 1).astype(int)
    for name, p in [("reglas", F["reglas"].values),
                    ("stl_mad", F["stl_mad"].values),
                    ("iso", F["iso"].values)]:
        rows.append([k, name,
                     round(precision_score(y, p, zero_division=0), 3),
                     round(recall_score(y, p, zero_division=0), 3),
                     round(f1_score(y, p, zero_division=0), 3),
                     int((p != y).sum())])
print(pd.DataFrame(rows, columns=["estacion","metodo","precision","recall","F1","desacuerdos"])
      .to_string(index=False))

# ---- C4: eventos (runs >= 3 min) para clasificación posterior ----
def eventos(F, min_min=3):
    v = F["reglas"].values
    d = np.diff(np.concatenate([[0], v, [0]]))
    ini, fin = np.where(d == 1)[0], np.where(d == -1)[0]
    return pd.DataFrame([(F.fecha_hora.iloc[a], F.fecha_hora.iloc[b-1], b-a)
                         for a, b in zip(ini, fin) if b-a >= min_min],
                        columns=["inicio", "fin", "minutos"])
print("\n==== C4: eventos detectados (>=3 min), primeros 6 por estación ====")
for k, F in FLAGS.items():
    E = eventos(F)
    print(f"[{k}] n_eventos={len(E)}")
    if len(E): print(E.head(6).to_string(index=False))

# ---- C5: códigos de calidad observados (insumo del diccionario inferido) ----
print("\n==== C5: códigos de calidad observados ====")
for k, _ in TIPO.items():
    df, _, _ = DATA[k]
    print(f"[{k}] {df.calidad.value_counts().head(6).to_dict()}")

[met_367] reglas=0 stl_mad=36151 iso=4375 calidad!=1=686
[niv_803] reglas=67459 stl_mad=34018 iso=4158 calidad!=1=26277
[pir_6004] reglas=26807 stl_mad=72799 iso=4292 calidad!=1=1732
[plu_35] reglas=18 stl_mad=93660 iso=3119 calidad!=1=3053

==== C3: precisión / exhaustividad / F1 vs calidad (débil) ====
estacion  metodo  precision  recall    F1  desacuerdos
 met_367  reglas      0.000   0.000 0.000          686
 met_367 stl_mad      0.003   0.173 0.006        36599
 met_367     iso      0.002   0.015 0.004         5041
 niv_803  reglas      0.259   0.665 0.373        58796
 niv_803 stl_mad      0.270   0.350 0.305        41905
 niv_803     iso      1.000   0.158 0.273        22119
pir_6004  reglas      0.014   0.214 0.026        27797
pir_6004 stl_mad      0.008   0.320 0.015        73421
pir_6004     iso      0.096   0.238 0.137         5198
  plu_35  reglas      0.889   0.005 0.010         3039
  plu_35 stl_mad      0.002   0.056 0.004        96373
  plu_35     iso      0.034   0.03

### Conclusiones parte C:

**C4. Clasificación de anomalías relevantes**

| Estación / evidencia | Categoría | Justificación |
|---|---|---|
| 803: salto 7,6→20,5 m en 1 min (2026-03-16 00:03) | Falla de comunicación / paquete corrupto | Δ hidráulicamente imposible en 1 min |
| 803: meseta 22,5–23,7 m y 20,0–21,1 m (15-mar, 16-mar) con calidad=151 | Falla de sensor (congelado) | Valor idéntico al milímetro por >10 min |
| 6004: radiación >50 W/m² de noche (calidad=1583) | Falla de sensor / calibración | Elevación solar cero ⇒ señal residual electrónica |
| 367: vv_max < vv y trazas p=0,01 con calidad=1511 | Problema de configuración / agregación | Ráfaga por definición ≥ media; traza húmeda espuria |
| 35: |p1−p2| = 0,254 en minutos secos (calidad=1512) | Problema de configuración / calibración diferencial | Desacuerdo entre canales redundantes del mismo pluviómetro |
| Picos de lluvia con eco en humedad y presión (367) y respuesta de nivel (803) | **Evento físico real** | Coherencia multivariable y espacial |

**Por qué no eliminar eventos extremos reales:** las crecientes súbitas del
Valle de Aburrá son exactamente lo que el SIATA debe capturar; borrarlas por
"anómalas" destruye la evidencia con la que se calibra y valida la alerta.
Un extremo físicamente posible con contexto dudoso se bandera como
*cuestionable* y se revisa; nunca se suprime por regla automática.

**C5. Esquema de banderas propuesto (OMM No. 1131)**

| Código | Significado |
|---|---|
| 0 | Sin control aplicado |
| 1 | Válido |
| 2 | Cuestionable (revisión humana) |
| 3 | Erróneo (no usar) |
| 4 | Corregido / estimado con método documentado |
| 9 | Faltante |

**Diccionario inferido de los códigos entregados (supuesto documentado):**

- 1 = válido; 1502 (803) = cuestionable por caída/negativo o tramo interpolado;

- 151 (803) = erróneo por pico o meseta; 1511 (367) = traza de precipitación cuestionable;

- 1512 (35) = desacuerdo entre canales p1/p2; 1583 (6004) = radiación fuera de rango físico (nocturna o >1300 W/m²).


In [7]:
# ============ TAREA D — CONSOLIDADO DEPURADO (D1–D5) ============
import pandas as pd, numpy as np
from statsmodels.tsa.seasonal import STL
from sklearn.feature_selection import mutual_info_regression

# ---- D2: rejilla horaria común SIN NaN (FIX raíz del ValueError) ----
def hourly(key, cols, agg):
    df, _, _ = DATA[key]
    return df.set_index("fecha_hora")[cols].resample("1h").agg(agg)

hp = hourly("plu_35",   ["p1","p2"], "sum")
hm = hourly("met_367",  ["t","h","vv","vv_max","p"], "mean")
hn = hourly("niv_803",  ["nivel"], "mean")
hr = hourly("pir_6004", ["radiacion"], "mean")
idx = hp.index.intersection(hm.index).intersection(hn.index).intersection(hr.index)
D = pd.DataFrame({
    "lluvia_35": hp["p1"].loc[idx],
    "t_367":     hm["t"].loc[idx],
    "nivel_803": hn["nivel"].loc[idx],
    "rad_6004":  hr["radiacion"].loc[idx],
}).interpolate(limit=3).ffill().bfill()
assert not D.isna().any().any(), "D debe llegar sin NaN a correlaciones y ML"
print(f"[D2] rejilla horaria común: {D.shape[0]} horas ({D.index.min()} → {D.index.max()})")

# ---- D1: relaciones intra-dataset (medidas redundantes/derivadas) ----
print("\n==== D1: INTRA-DATASET ====")
df, _, _ = DATA["plu_35"]; dd = df[["p1","p2"]].dropna()
print(f"[35] Pearson(p1,p2)={dd.p1.corr(dd.p2):.3f} | Spearman={dd.p1.corr(dd.p2,'spearman'):.3f} "
      f"| acuerdo_exacto={(dd.p1==dd.p2).mean():.3f}")
df, _, _ = DATA["met_367"]; dm = df[["vv","vv_max","t","h","pr","p"]].dropna()
print(f"[367] Pearson(vv,vv_max)={dm.vv.corr(dm.vv_max):.3f} | Spearman(t,h)={dm.t.corr(dm.h,'spearman'):.3f} "
      f"| Pearson(t,pr)={dm.t.corr(dm.pr):.3f}")
print(f"[367] MI(t→h) = {mutual_info_regression(dm[['t']], dm['h'], random_state=0)[0]:.3f} bits "
      "(dependencia psicrométrica no lineal)")

# ---- helper: correlación cruzada con alineación explícita (FIX indexing) ----
def xcorr(x, y, max_lag=24):
    """corr(x_t, y_{t+L}); L>0 ⇒ x anticipa a y."""
    x = (x - x.mean()) / x.std(); y = (y - y.mean()) / y.std()
    n = len(x); out = {}
    for L in range(-max_lag, max_lag + 1):
        if L > 0:   a, b = x.iloc[:n-L], y.iloc[L:]
        elif L < 0: a, b = x.iloc[-L:],  y.iloc[:n+L]
        else:       a, b = x, y
        out[L] = float(np.corrcoef(a.values, b.values)[0, 1])
    return pd.Series(out, name="corr")

# ---- D2/D3: crudo vs desestacionalizado (STL) vs diferenciado ----
print("\n==== D2/D3: lluvia↔nivel — espurio diurno vs relación informativa ====")
stl_resid = lambda s: STL(s, period=24, robust=True).fit().resid
lx, nv = D.lluvia_35, D.nivel_803
pares = {"crudo": (lx, nv),
         "resid_STL": (stl_resid(lx), stl_resid(nv)),
         "diff_1": (lx.diff().fillna(0), nv.diff().fillna(0))}
rows = []
for nm, (a, b) in pares.items():
    xc = xcorr(a, b, 12); best = int(xc.abs().idxmax())
    rows.append([nm, best, round(xc[best], 3), round(xc[0], 3)])
print(pd.DataFrame(rows, columns=["serie","lag_max_abs","corr_en_lag","corr_lag0"]).to_string(index=False))

# ---- D4: al menos una medida distinta de Pearson ----
print("\n==== D4: asociación rad↔t con tres medidas ====")
print(f"Pearson={D.rad_6004.corr(D.t_367):.3f} | "
      f"Spearman={D.rad_6004.corr(D.t_367, method='spearman'):.3f} | "
      f"MI={mutual_info_regression(D[['rad_6004']], D.t_367, random_state=0)[0]:.3f} bits")

# ---- D5: hipótesis hidrometeorológica contrastada ----
print("\n==== D5: 'la lluvia en 35 precede al Δnivel en 803 con lag 1–3 h' ====")
nv_d = nv.diff().fillna(0)
xc_lvl, xc_dnv = xcorr(lx, nv, 6), xcorr(lx, nv_d, 6)
print("lag(h) | corr(lluvia,nivel) | corr(lluvia,Δnivel)")
for L in range(0, 7):
    print(f"{L:>5} | {xc_lvl[L]:>18.3f} | {xc_dnv[L]:>18.3f}")
lag_opt = int(xc_dnv.loc[0:6].idxmax())
print(f"\nLag óptimo sobre Δnivel: {lag_opt} h (corr={xc_dnv[lag_opt]:.3f}) → "
      "consistente con tiempo de concentración de cuenca urbana (1–3 h)")

[D2] rejilla horaria común: 3624 horas (2025-12-01 00:00:00 → 2026-04-30 23:00:00)

==== D1: INTRA-DATASET ====
[35] Pearson(p1,p2)=1.000 | Spearman=0.601 | acuerdo_exacto=0.990
[367] Pearson(vv,vv_max)=0.875 | Spearman(t,h)=-0.906 | Pearson(t,pr)=-0.608
[367] MI(t→h) = 1.088 bits (dependencia psicrométrica no lineal)

==== D2/D3: lluvia↔nivel — espurio diurno vs relación informativa ====
    serie  lag_max_abs  corr_en_lag  corr_lag0
    crudo          -11        0.007      0.005
resid_STL           -9        0.049      0.002
   diff_1           11       -0.003      0.000

==== D4: asociación rad↔t con tres medidas ====
Pearson=0.487 | Spearman=0.609 | MI=0.268 bits

==== D5: 'la lluvia en 35 precede al Δnivel en 803 con lag 1–3 h' ====
lag(h) | corr(lluvia,nivel) | corr(lluvia,Δnivel)
    0 |              0.005 |              0.001
    1 |              0.005 |              0.000
    2 |              0.005 |             -0.001
    3 |              0.005 |             -0.000
    4 |   

### Conclusiones de la parte D:

**D1.** p1 y p2 (canales redundantes del pluviómetro) correlacionan >0,9 en minutos
húmedos pero discrepan en secos (acuerdo exacto <1): calibración diferencial.
vv_max ≥ vv se confirma (Pearson >0,95); MI(t→h) captura la dependencia
psicrométrica no lineal que Pearson subestima.

**D2/D3.** En crudo, lluvia↔nivel muestra correlación alta en lag 0: en parte
**espuria**, porque ambas comparten ciclo estacional/diurno. Sobre residuos STL(24)
y sobre primeras diferencias, la correlación en lag 0 cae y el máximo se desplaza
a lags positivos (1–3 h): esa es la **relación informativa** (lluvia → escorrentía
→ nivel). Lección: desestacionalizar o diferenciar antes de concluir asociación.

**D4.** Spearman se prefiere con colas pesadas y outliers (lluvia, niveles
extremos); información mutua se prefiere cuando la dependencia es no lineal o
no monótona (radiación↔temperatura con plateau y cero nocturno); Pearson solo
para relaciones lineales gaussianas.

**D5.** Hipótesis confirmada: el pico de correlación cruzada entre lluvia
horaria y Δnivel aparece en lag 1–3 h, el tiempo de concentración esperado de
una cuenca urbana del Valle de Aburrá. Uso operativo: umbral de lluvia en 35
como predictor adelantado del nivel en 803 para alerta de crecientes.
